In [1]:
from IPython.display import HTML, Markdown, display
import plotly.graph_objects as go
import pandas as pd

def display_eval_report(eval_result: pd.DataFrame) -> None:
    """Display the evaluation results."""
    metrics_df = pd.DataFrame.from_dict(eval_result.summary_metrics, orient="index").T
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    display(Markdown(f"### Row-wise Metrics"))
    display(eval_result.metrics_table)
    
prompt = [
    "Turn device_2 power off", # example 1
    "Get user_x preference temperature and set Living Room temperature to the preferred value", # example 2
    "Get user_y preference temperature and set Master Room temperature to the preferred value", # example 3
    "Set all devices off" # example 4
]

reference_trajectory = [
# example 1
[
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_2",
        "updates": {
            "status": "OFF"
        }
    }
  }
],
# example 2
[
    {
      "tool_name": "get_user_preferences",
      "tool_input": {
          "user_id": "user_x"
      }
  },
  {
      "tool_name": "set_temperature",
      "tool_input": {
          "location": "Living Room",
          "temperature": 23
      }
    },
],
# example 3
[
    {
      "tool_name": "get_user_preferences",
      "tool_input": {
          "user_id": "user_y"
      }
  },
  {
      "tool_name": "set_temperature",
      "tool_input": {
          "location": "Master Room",
          "temperature": 26
      }
    },
],
# example 4
[
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_1",
        "updates": {
            "status": "OFF"
        }
    }
  },
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_2",
        "updates": {
            "status": "OFF"
        }
    }
  }
]
]

predicted_trajectory = [
# example 1
[
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_3",
        "updates": {
            "status": "OFF"
        }
    }
  }
],
# example 2
[
    {
      "tool_name": "get_user_preferences",
      "tool_input": {
          "user_id": "user_z"
      }
    },
    {
      "tool_name": "set_temperature",
      "tool_input": {
          "location": "Living Room",
          "temperature": 23
      }
    },
],
# example 3, does not care about input parameter order
[
    {
      "tool_name": "get_user_preferences",
      "tool_input": {
          "user_id": "user_y"
      }
  },
  {
      "tool_name": "set_temperature",
      "tool_input": {          
          "temperature": 26,
          "location": "Master Room"
      }
    },
],
# example 4, add additional device in route
[
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_2",
        "updates": {
            "status": "OFF"
        }
    }
  },
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_1",
        "updates": {
            "status": "OFF"
        }
    }
  },
  {
    "tool_name": "set_device_info",
    "tool_input": {
        "device_id": "device_3",
        "updates": {
            "status": "OFF"
        }
    }
  }
]
]

response = [
    "Device 3 power off",
    "Set Living Room temperature to 23 celcius",
    "Set Master Room temperature to 26 celcius",
    "All devices turned off"
]

eval_dataset = pd.DataFrame({
    "prompt": prompt,
    "predicted_trajectory": predicted_trajectory,
    "reference_trajectory": reference_trajectory,
    "response": response
})
eval_dataset

,prompt,predicted_trajectory,reference_trajectory,response
0,Turn device_2 power off,"[{'tool_name': 'set_device_info', 'tool_input'...","[{'tool_name': 'set_device_info', 'tool_input'...",Device 3 power off
1,Get user_x preference temperature and set Livi...,"[{'tool_name': 'get_user_preferences', 'tool_i...","[{'tool_name': 'get_user_preferences', 'tool_i...",Set Living Room temperature to 23 celcius
2,Get user_y preference temperature and set Mast...,"[{'tool_name': 'get_user_preferences', 'tool_i...","[{'tool_name': 'get_user_preferences', 'tool_i...",Set Master Room temperature to 26 celcius
3,Set all devices off,"[{'tool_name': 'set_device_info', 'tool_input'...","[{'tool_name': 'set_device_info', 'tool_input'...",All devices turned off


In [ ]:
from vertexai.preview.evaluation import EvalTask
import google.auth
import vertexai
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
vertexai.init(project=PROJECT_ID, location=LOCATION)
eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=[
        "trajectory_exact_match", # check exactly same 0/1
        "trajectory_in_order_match", # check order matched and have extra functions
        "trajectory_any_order_match", # check order not matched and have extra functions
        "trajectory_precision", #0-1, higher is better, (count(predicted found in reference))/(total number of actions in predicted)
        "trajectory_recall", #0-1, higher is better, (count(reference found in predicted))/(total number of actions in reference)
    ],
)

#Use runnable if dynamic generation required, this will generates latency and failure parts
eval_result = eval_task.evaluate(
    #runnable=RUNNABLE,
)

display_eval_report(eval_result)

Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 20/20 [00:03<00:00,  5.62it/s]

All 20 metric requests are successfully computed.
Evaluation Took:3.571740699990187 seconds


### Summary Metrics

,row_count,trajectory_exact_match/mean,trajectory_exact_match/std,trajectory_in_order_match/mean,trajectory_in_order_match/std,trajectory_any_order_match/mean,trajectory_any_order_match/std,trajectory_precision/mean,trajectory_precision/std,trajectory_recall/mean,trajectory_recall/std
0,4.0,0.25,0.5,0.25,0.5,0.5,0.57735,0.541667,0.416667,0.625,0.478714


### Row-wise Metrics

,prompt,predicted_trajectory,reference_trajectory,response,trajectory_exact_match/score,trajectory_in_order_match/score,trajectory_any_order_match/score,trajectory_precision/score,trajectory_recall/score
0,Turn device_2 power off,"[{'tool_name': 'set_device_info', 'tool_input'...","[{'tool_name': 'set_device_info', 'tool_input'...",Device 3 power off,0.0,0.0,0.0,0.000000,0.0
1,Get user_x preference temperature and set Livi...,"[{'tool_name': 'get_user_preferences', 'tool_i...","[{'tool_name': 'get_user_preferences', 'tool_i...",Set Living Room temperature to 23 celcius,0.0,0.0,0.0,0.500000,0.5
2,Get user_y preference temperature and set Mast...,"[{'tool_name': 'get_user_preferences', 'tool_i...","[{'tool_name': 'get_user_preferences', 'tool_i...",Set Master Room temperature to 26 celcius,1.0,1.0,1.0,1.000000,1.0
3,Set all devices off,"[{'tool_name': 'set_device_info', 'tool_input'...","[{'tool_name': 'set_device_info', 'tool_input'...",All devices turned off,0.0,0.0,1.0,0.666667,1.0
